# 🎬 สรุปคลิป — วางลิงก์ YouTube / TikTok / Facebook แล้วได้คลิปสรุปพากย์ไทย (ใช้บนมือถือได้)

**วิธีใช้ (ครั้งแรกใช้เวลา 3–8 นาที)**
1. แนะนำเปิดการ์ดจอฟรีก่อน: เมนู **Runtime → Change runtime type → T4 GPU** → Save
2. เลือกการตั้งค่าในช่องด้านล่าง
3. กด **Runtime → Run all** (บนมือถืออยู่ในเมนู ⋮ หรือ ☰)
4. รอจนขึ้นปุ่มสีส้ม **"เปิดโปรแกรม"** แล้วแตะเพื่อใช้งาน
5. วางลิงก์คลิป → กด **เริ่มสรุปคลิป** → รอรับวิดีโอ

🎮 **ไฮไลต์เกม MLBB:** เลือกโหมด "ไฮไลต์เกม MLBB" ในโปรแกรม แล้วใส่คลิปอัดหน้าจอ (ต้องมีเสียงเกม) → ได้คลิปแนวตั้งพร้อมป้าย SAVAGE / MANIAC สโลว์โมชัน และเสียงพากย์แคสเตอร์
📝 **ข้อความ → คลิป:** พิมพ์/วางบท → พากย์ตามบททุกคำ (หรือให้ AI เรียบเรียง) แบ่งฉาก วาดภาพ (ครั้งแรกโหลดโมเดลวาดภาพ ~7 GB รอ 2-5 นาที) ทำแอนิเมชัน และพากย์ให้ — ต้องเปิด T4 GPU
🎨 เปิด "ตกแต่งคลิป" เพื่อใส่เพลง ชื่อช่อง โลโก้ และภาพปก · ใส่หลายลิงก์/หลายไฟล์ได้ (แยกคลิปทุกเกม หรือรวมช็อตเด็ดเป็นคลิปเดียว)

⚠️ เปิดหน้านี้ค้างไว้ระหว่างใช้งาน ถ้าปิดหรือไม่ได้ใช้นาน Colab จะหยุดเครื่อง และไฟล์จะหายไป (เว้นแต่เลือกบันทึกลง Google Drive)

In [ ]:
#@title ⚙️ ตั้งค่า
#@markdown **ใช้อะไรสรุปเนื้อหา** — `ollama` = ฟรี (ดาวน์โหลดโมเดลเพิ่ม ~5 GB ครั้งแรก) · `claude` = สรุปดีที่สุด แต่จ่ายตามการใช้ · `none` = เขียนบทพากย์เอง
AI = "ollama"  #@param ["ollama", "claude", "none"]
#@markdown คีย์ Claude API (ใช้เมื่อเลือก `claude`) — หรือเก็บไว้ใน 🔑 Secrets ชื่อ `ANTHROPIC_API_KEY` จะปลอดภัยกว่า
ANTHROPIC_API_KEY = ""  #@param {type:"string"}
#@markdown บันทึกงานและวิดีโอลง Google Drive (โฟลเดอร์ "สรุปคลิป") — ไฟล์ไม่หายเมื่อ Colab ปิด
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
#@markdown สร้างลิงก์สำรอง (ใช้เมื่อปุ่มหลักเปิดไม่ได้ เช่นบน iPhone/Safari)
BACKUP_LINK = True  #@param {type:"boolean"}


In [ ]:
#@title 🚀 เริ่มโปรแกรม — รอจนขึ้นปุ่ม "เปิดโปรแกรม"
import base64, io, os, re, subprocess, sys, time, urllib.request, zipfile
from IPython.display import HTML, display

APP_ZIP = "..."

def step(t): print(t, flush=True)
def wait_url(url, sec):
    for _ in range(sec):
        try:
            urllib.request.urlopen(url, timeout=2); return True
        except Exception:
            time.sleep(1)
    return False

os.chdir("/content")
zipfile.ZipFile(io.BytesIO(base64.b64decode(APP_ZIP))).extractall("/content")
APP = "/content/video-summarizer"
subprocess.run("pkill -f 'python app.py'; pkill -f cloudflared", shell=True)

step("1/4 ติดตั้งไลบรารี…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "-r", f"{APP}/requirements.txt"], check=True)
# วาดภาพด้วย AI (โหมดข้อความ → คลิป) — torch มีใน Colab อยู่แล้ว
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers", "accelerate"], capture_output=True)
# ฟอนต์ไทย สำหรับซับไตเติล ชื่อช่อง และภาพปก (Colab ไม่มีมาให้)
subprocess.run("apt-get install -y -qq fonts-thai-tlwg > /dev/null 2>&1", shell=True)
# ตัวรัน JavaScript ที่ yt-dlp ต้องใช้กับ YouTube (ถ้าติดตั้งไม่ได้ จะใช้ node ที่มีใน Colab แทน)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deno"], capture_output=True)

gpu = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
step("   การ์ดจอ: " + ("มี ✓ (ถอดเสียงเร็ว)" if gpu else "ไม่มี — ถอดเสียงจะช้า แนะนำเปลี่ยนเป็น T4 GPU"))

env = os.environ.copy()
env["HOST"], env["PORT"] = "127.0.0.1", "8000"
key = ANTHROPIC_API_KEY.strip()
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("ANTHROPIC_API_KEY") or ""
    except Exception:
        key = ""
if key:
    env["ANTHROPIC_API_KEY"] = key
elif AI == "claude":
    step("   ⚠️ ยังไม่ได้ใส่คีย์ Claude — จะใช้โหมดเขียนบทเอง")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    env["JOBS_DIR"] = "/content/drive/MyDrive/สรุปคลิป"

if AI == "ollama":
    step("2/4 ติดตั้ง Ollama และโหลดโมเดล AI (ครั้งแรก 2–5 นาที)…")
    if subprocess.run("which ollama", shell=True, capture_output=True).returncode != 0:
        subprocess.run("apt-get install -y -qq zstd > /dev/null; curl -fsSL https://ollama.com/install.sh | sh > /dev/null",
                       shell=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if wait_url("http://127.0.0.1:11434/api/tags", 60):
        subprocess.run(["ollama", "pull", "qwen2.5:7b"], stdout=subprocess.DEVNULL)
    else:
        step("   ⚠️ เปิด Ollama ไม่สำเร็จ — จะใช้โหมดเขียนบทเอง")
else:
    step("2/4 ข้ามการติดตั้ง Ollama")

step("3/4 เปิดโปรแกรม…")
log = open("/content/app.log", "w")
subprocess.Popen([sys.executable, "app.py"], cwd=APP, env=env, stdout=log, stderr=subprocess.STDOUT)
if not wait_url("http://127.0.0.1:8000/api/config", 60):
    print(open("/content/app.log").read()); raise SystemExit("โปรแกรมเปิดไม่สำเร็จ ดูข้อความด้านบน")

step("4/4 สร้างลิงก์…")
from google.colab.output import eval_js
main_url = eval_js("google.colab.kernel.proxyPort(8000, {'cache': false})")
backup = ""
if BACKUP_LINK:
    if not os.path.exists("/content/cloudflared"):
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                                   "/content/cloudflared")
        os.chmod("/content/cloudflared", 0o755)
    cf = open("/content/cloudflared.log", "w")
    subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                     stdout=cf, stderr=cf)
    for _ in range(40):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
        if m:
            backup = m.group(0); break
        time.sleep(1)

btn = "display:block;text-align:center;padding:16px;border-radius:12px;font:600 18px sans-serif;text-decoration:none;margin:10px 0"
html = f'<a href="{main_url}" target="_blank" style="{btn};background:#d9480f;color:#fff">▶ เปิดโปรแกรม</a>'
if backup:
    html += (f'<a href="{backup}" target="_blank" style="{btn};border:2px solid #d9480f;color:#d9480f">ลิงก์สำรอง (ถ้าปุ่มบนเปิดไม่ได้)</a>'
             '<div style="font:13px sans-serif;color:#888">ลิงก์สำรองใครมีลิงก์ก็เปิดได้ อย่าแชร์ให้คนอื่น · ลิงก์เปลี่ยนทุกครั้งที่รันใหม่</div>')
display(HTML(html))
print("\n✅ พร้อมใช้งาน — เปิดหน้านี้ค้างไว้ระหว่างใช้ (กดปุ่ม ■ หยุดเซลล์นี้เมื่อเลิกใช้)")

# วนรอไว้เพื่อให้ Colab รู้ว่ายังใช้งานอยู่ และแจ้งถ้าโปรแกรมล่ม
try:
    while True:
        time.sleep(60)
        if not wait_url("http://127.0.0.1:8000/api/config", 5):
            print("⚠️ โปรแกรมหยุดทำงาน:\n" + open("/content/app.log").read()[-2000:]); break
except KeyboardInterrupt:
    subprocess.run("pkill -f 'python app.py'; pkill -f cloudflared; pkill -f 'ollama serve'", shell=True)
    print("ปิดโปรแกรมแล้ว")
